In [100]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

# Pfad zur CSV-Datei — funktioniert sowohl aus dem Notebook-Ordner
# als auch aus dem Projektstamm heraus.
csv_path = Path.cwd() / "data" / "[OEECore.PenzDia.DW].[dbo].[vw_batch_kpis].csv"
if not csv_path.exists():
    csv_path = Path.cwd().parent / "data" / "[OEECore.PenzDia.DW].[dbo].[vw_batch_kpis].csv"

# Die Datei ist semicolon-getrennt, verwendet das deutsche Dezimalkomma
# und enthält einen Header in der ersten Zeile.
df = pd.read_csv(
    csv_path,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    low_memory=False,
)

print("Shape:", df.shape)
df.head(3)

Shape: (578, 61)


,oee1,oee2,oee3,availability,performance,quality,capacity_utilization,oae,theoretical_run_rate,actual_run_rate,...,not_scheduled_time_min,unscheduled_time_min,uptime_min,line_status_duration_min,alarm_duration_min,alarm_count,line_status_count,upd_ts,site_oid,line_oid
0,0.448913,0.448913,NaN,0.483851,0.955222,0.971285,1.0,0.462185,200.0,191.044382,...,NaN,NaN,686.489698,1418.804983,0,0,901,2025-07-12 03:20:44.000,F07AFA54-6864-41C7-8CF0-F4F92F7B7C4F,C8EB00B5-3D04-47E8-9B32-96D06816C51C
1,0.431932,0.431932,NaN,0.659754,0.674915,0.970027,1.0,0.445278,200.0,134.983082,...,NaN,NaN,49.680300,75.301249,0,0,135,2025-09-24 17:32:22.000,F07AFA54-6864-41C7-8CF0-F4F92F7B7C4F,C8EB00B5-3D04-47E8-9B32-96D06816C51C
2,0.479215,0.479215,NaN,0.645979,0.756515,0.980607,1.0,0.488692,200.0,151.302916,...,NaN,NaN,61.003451,94.435667,0,0,87,2025-10-15 16:49:45.000,F07AFA54-6864-41C7-8CF0-F4F92F7B7C4F,C8EB00B5-3D04-47E8-9B32-96D06816C51C


## 1 · Datenaufbereitung

In [101]:

# Linie und Gerät identifizieren.
# Diese Datei enthält nur eine Linie und eine line_oid — zur Sicherheit
# wird trotzdem explizit gefiltert, damit der Code stabil bleibt,
# wenn die Quelldatei künftig mehrere Linien enthalten sollte.
LINE_NAME  = "Abfüllanlage 4-E2G"
LINE_OID   = "C8EB00B5-3D04-47E8-9B32-96D06816C51C"
LINE_LABEL = "Abfüllanlage 4-E2G"

# oee2 — reiner Produktions-OEE ohne Kapazitätsfaktor.
# Bei capacity_utilization = 1 ist oee2 identisch mit oee1.
# Bei Teilauslastung weicht oee1 nach unten ab, oee2 bleibt unbeeinflusst.
OEE_COL = "oee2"

df_line = df[(df["line_name"] == LINE_NAME) & (df["line_oid"] == LINE_OID)].copy()

# Numerische Spalten absichern.
for col in [OEE_COL, "availability", "performance", "quality", "total_quantity",
           "actual_operating_time_min"]:
    df_line[col] = pd.to_numeric(df_line[col], errors="coerce")

# Start und Stop sind in dieser Datei bereits kombinierte Datetime-Felder —
# kein Zusammenführen zweier Spalten wie in der Cobas-Datei nötig.
df_line["start"]   = pd.to_datetime(df_line["start"], errors="coerce")
df_line["stop_dt"] = pd.to_datetime(df_line["stop"],  errors="coerce")

n_before_dropna = len(df_line)
df_line = df_line.dropna(subset=[OEE_COL, "start", "stop_dt"])
n_dropna = n_before_dropna - len(df_line)
if n_dropna > 0:
    print(f"  {n_dropna} Charge(n) entfernt: fehlende Werte (NaN in {OEE_COL}/start/stop).")

# Chargen ohne Produktionsmenge entfernen.
n_zero = (df_line["total_quantity"] == 0).sum()
df_line = df_line[df_line["total_quantity"] > 0].copy()
if n_zero > 0:
    print(f"  {n_zero} Charge(n) entfernt: keine Produktionsmenge (Leerläufe).")

# Ungültige OEE-Werte entfernen.
n_invalid = ((df_line[OEE_COL] <= 0) | (df_line[OEE_COL] > 1)).sum()
df_line = df_line[(df_line[OEE_COL] > 0) & (df_line[OEE_COL] <= 1)].copy()
if n_invalid > 0:
    print(f"  {n_invalid} Charge(n) entfernt: ungültiger OEE-Wert.")

# OEE auf Prozent skalieren (clip schützt vor Floating-Point-Ungenauigkeiten).
df_line["oee_pct"] = (df_line[OEE_COL] * 100).clip(upper=100)
# Betriebsdauer aus dem Quellsystem übernehmen (actual_operating_time_min).
df_line["duration_min"] = df_line["actual_operating_time_min"]
n_no_dur = (df_line["duration_min"].isna() | (df_line["duration_min"] <= 0)).sum()
df_line = df_line[df_line["duration_min"] > 0].copy()
if n_no_dur > 0:
    print(f"  {n_no_dur} Charge(n) entfernt: keine gültige Betriebsdauer (actual_operating_time_min ≤ 0 oder NaN).")

# OEE-Komponenten auf Prozent skalieren und auf [0, 100] begrenzen.
# Werte über 100 % entstehen durch zu niedrig hinterlegte Sollwerte
# im Quellsystem (Stammdatenfehler) — sie werden gekappt, nicht entfernt.
for col_raw, col_pct in [("availability", "availability_pct"),
                          ("performance",  "performance_pct"),
                          ("quality",       "quality_pct")]:
    df_line[col_pct] = (pd.to_numeric(df_line[col_raw], errors="coerce") * 100
                        ).clip(lower=0, upper=100)

# Für den Zeitreihenverlauf auf Monatsebene aggregieren.
df_line["month"] = df_line["start"].dt.to_period("M")

# Zeitgewichteter monatlicher Mittelwert:
# np.average(..., weights=duration_min) gewichtet jeden OEE-Wert
# proportional zur Chargendauer.
monthly = (
    df_line.groupby("month")
    .apply(lambda x: pd.Series({
        "mean":  np.average(x["oee_pct"], weights=x["duration_min"]),
        "std":   x["oee_pct"].std(),
        "count": len(x),
    }))
    .reset_index()
)
monthly["month_dt"] = monthly["month"].dt.to_timestamp()
monthly["std"]   = monthly["std"].fillna(0)
monthly["count"] = monthly["count"].astype(int)

# Zeitgewichteter Gesamtmittelwert.
overall_mean = np.average(df_line["oee_pct"], weights=df_line["duration_min"])

print(f"\nDatensatz {LINE_LABEL}: {len(df_line)} Chargen")
print(f"Zeitraum:  {monthly['month'].min()} – {monthly['month'].max()}  ({len(monthly)} Monate)")
print(f"Ø OEE ({OEE_COL}, zeitgewichtet): {overall_mean:.1f} %")
print(f"Ø Verfügbarkeit (zeitgew.):  {np.average(df_line['availability_pct'], weights=df_line['duration_min']):.1f} %")
print(f"Ø Leistungsgrad (zeitgew.):  {np.average(df_line['performance_pct'],  weights=df_line['duration_min']):.1f} %")
print(f"Ø Qualität      (zeitgew.):  {np.average(df_line['quality_pct'],      weights=df_line['duration_min']):.1f} %")
# print(f"Ø Leistungsgrad (zeitgew.):  {np.average(df_line['performance_pct'],  weights=df_line['duration_min']):.1f} %")                ALTER CODE VOR OEE ZU MOANAT KPIS

# ── NEU: Monatliche System-KPIs aus vw_monthly_kpis laden ──────────────────────
# Wird in Zelle 5 (OEE-Verlauf) als Referenzlinie verwendet.
# Ohne diesen Block → System-OEE-Linie in Zelle 5 entfernen.
monthly_kpi_path = csv_path.parent / "[OEECore.PenzDia.DW].[dbo].[vw_monthly_kpis].csv"
monthly_sys = pd.read_csv(monthly_kpi_path, sep=";", decimal=",", encoding="utf-8-sig", low_memory=False)
monthly_sys = monthly_sys[monthly_sys["line_oid"] == LINE_OID].copy()
monthly_sys["month_dt"] = pd.to_datetime(monthly_sys["month_key"])
for _c in ["oee2", "availability", "performance", "quality"]:
    monthly_sys[_c] = pd.to_numeric(monthly_sys[_c], errors="coerce")
monthly_sys["oee_sys"]   = (monthly_sys["oee2"]         * 100).clip(upper=100)
monthly_sys["avail_sys"] = (monthly_sys["availability"]  * 100).clip(0, 100)
monthly_sys["perf_sys"]  = (monthly_sys["performance"]   * 100).clip(0, 100)
monthly_sys["qual_sys"]  = (monthly_sys["quality"]       * 100).clip(0, 100)
monthly_sys = monthly_sys.sort_values("month_dt").reset_index(drop=True)

overall_sys = monthly_sys["oee_sys"].mean()

print(f"\nSystem-KPIs (vw_monthly_kpis): {len(monthly_sys)} Monate")
print(f"Ø OEE (System): {overall_sys:.1f} %")

  41 Charge(n) entfernt: fehlende Werte (NaN in oee2/start/stop).
  10 Charge(n) entfernt: keine Produktionsmenge (Leerläufe).
  21 Charge(n) entfernt: ungültiger OEE-Wert.

Datensatz Abfüllanlage 4-E2G: 506 Chargen
Zeitraum:  2025-01 – 2026-03  (15 Monate)
Ø OEE (oee2, zeitgewichtet): 44.3 %
Ø Verfügbarkeit (zeitgew.):  50.1 %
Ø Leistungsgrad (zeitgew.):  84.7 %
Ø Qualität      (zeitgew.):  96.8 %

System-KPIs (vw_monthly_kpis): 15 Monate
Ø OEE (System): 32.3 %


## 2 · Monatlicher OEE-Verlauf

In [102]:
import plotly.graph_objects as go

x      = monthly["month_dt"]
y_mean = monthly["mean"]
y_std  = monthly["std"]

fig = go.Figure()

# Einzelchargen als sehr kleine, halbtransparente Punkte im Hintergrund.
fig.add_trace(go.Scatter(
    x=df_line["start"], y=df_line["oee_pct"],
    mode="markers", name="Einzelchargen",
    marker=dict(color="#4C72B0", size=4, opacity=0.15),
    hovertemplate="Datum: %{x|%Y-%m-%d}<br>OEE: %{y:.1f} %<extra>Einzelcharge</extra>",
))

# ±1σ-Band als geschlossene Fläche (Plotly-Technik: Pfad vorwärts + rückwärts).
fig.add_trace(go.Scatter(
    x=list(x) + list(x[::-1]),
    y=list(y_mean + y_std) + list((y_mean - y_std)[::-1]),
    fill="toself", fillcolor="rgba(76, 114, 176, 0.18)",
    line=dict(color="rgba(255,255,255,0)"),
    hoverinfo="skip", showlegend=True, name="±1 Std.-Abweichung",
))

# Zeitgewichtete monatliche Mittelwertlinie.
fig.add_trace(go.Scatter(
    x=x, y=y_mean,
    mode="lines+markers", name="Monatl. Ø OEE",
    line=dict(color="#4C72B0", width=2.5),
    marker=dict(size=8, color="white", line=dict(color="#4C72B0", width=2)),
    hovertemplate=(
        "<b>%{x|%B %Y}</b><br>"
        "Ø OEE: %{y:.1f} %<br>"
        "σ: %{customdata[0]:.1f} %<br>"
        "Chargen: %{customdata[1]}<extra></extra>"
    ),
    customdata=np.column_stack([y_std, monthly["count"]]),
))

# Horizontale Referenzlinie für den zeitgewichteten Gesamtmittelwert.
fig.add_hline(
    y=overall_mean,
    line=dict(color="#C44E52", width=2, dash="dash"),
    annotation_text=f"Ø {overall_mean:.1f} %",
    annotation_position="top right",
    annotation_font=dict(color="#C44E52", size=12),
)

# ── NEU: System-OEE aus vw_monthly_kpis ──────────────────────────────────────
# Entfernen, falls der Block in Zelle 3 (monthly_sys) nicht mehr geladen wird.
fig.add_trace(go.Scatter(
    x=monthly_sys["month_dt"], y=monthly_sys["oee_sys"],
    mode="lines+markers", name="System-OEE (vw_monthly_kpis)",
    line=dict(color="#DD8452", width=2.5, dash="dot"),
    marker=dict(size=8, color="white", line=dict(color="#DD8452", width=2)),
    hovertemplate=(
        "<b>%{x|%B %Y}</b><br>"
        "System-OEE: %{y:.1f} %<extra>vw_monthly_kpis</extra>"
    ),
))
# ── ENDE NEU ─────────────────────────────────────────────────────────────────

fig.update_layout(
    title=dict(
        text=(
            f"Monatlicher OEE-Verlauf — {LINE_LABEL}<br>"
            f"<sup>{len(df_line)} Chargen in der Auswertung "
            f"| Mittelwert zeitgewichtet nach Chargendauer (OEE-Variante: {OEE_COL})</sup>"
        ),
        font=dict(size=16),
    ),
    xaxis=dict(title="Monat", tickformat="%b\n%Y", dtick="M1",
               showgrid=True, gridcolor="#e0e0e0"),
    yaxis=dict(title="OEE [%]", range=[0, 100],
               showgrid=True, gridcolor="#e0e0e0"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    width=1100, height=500, hovermode="x unified",
)
fig.show()


## 3 · OEE-Komponenten im Zeitverlauf

In [103]:
# Zeitgewichtete monatliche Komponentenaggregation.
pct_cols = ["oee_pct", "availability_pct", "performance_pct", "quality_pct"]
monthly_comp = (
    df_line.groupby("month")
    .apply(lambda x: pd.Series({
        col: np.average(x[col], weights=x["duration_min"])
        for col in pct_cols
    }))
    .reset_index()
)
monthly_comp["month_dt"] = monthly_comp["month"].dt.to_timestamp()

COMP = {
    "availability_pct": {"name": "Verfügbarkeit", "color": "#4C72B0", "dash": None,  "width": 2.2},
    "performance_pct":  {"name": "Leistungsgrad", "color": "#DD8452", "dash": None,  "width": 2.2},
    "quality_pct":      {"name": "Qualität",      "color": "#55A868", "dash": None,  "width": 2.2},
    "oee_pct":          {"name": "OEE (Ist)",     "color": "#C44E52", "dash": None,  "width": 3.0},
}

fig_comp = go.Figure()

# Einzelchargen als sehr kleine Punkte im Hintergrund (nur OEE).
fig_comp.add_trace(go.Scatter(
    x=df_line["start"], y=df_line["oee_pct"],
    mode="markers", name="Einzelchargen (OEE)",
    marker=dict(color="#C44E52", size=3, opacity=0.12),
    hovertemplate="Datum: %{x|%d.%m.%Y}<br>OEE: %{y:.1f} %<extra>Einzelcharge</extra>",
))

for comp_col, cfg in COMP.items():
    if comp_col not in monthly_comp.columns:
        continue
    line_cfg = dict(color=cfg["color"], width=cfg["width"])
    if cfg["dash"]:
        line_cfg["dash"] = cfg["dash"]
    fig_comp.add_trace(go.Scatter(
        x=monthly_comp["month_dt"], y=monthly_comp[comp_col],
        mode="lines+markers", name=cfg["name"],
        line=line_cfg,
        marker=dict(size=7, color="white", line=dict(color=cfg["color"], width=2)),
        hovertemplate=(
            f"<b>%{{x|%B %Y}}</b><br>{cfg['name']}: %{{y:.1f}} %<extra></extra>"
        ),
    ))

fig_comp.add_hline(
    y=overall_mean,
    line=dict(color="#C44E52", width=1.5, dash="dash"),
    annotation_text=f"Ø OEE: {overall_mean:.1f} %",
    annotation_position="top right",
    annotation_font=dict(color="#C44E52", size=12),
)

fig_comp.update_layout(
    title=dict(
        text=(
            f"OEE-Komponenten im Zeitverlauf — {LINE_LABEL}<br>"
            f"<sup>{len(df_line)} Chargen | Leistungsgrad- und Qualitätswerte auf 100 % begrenzt "
            f"| Mittelwert zeitgewichtet nach Chargendauer</sup>"
        ),
        font=dict(size=16),
    ),
    xaxis=dict(title="Monat", tickformat="%b\n%Y", dtick="M1",
               showgrid=True, gridcolor="#e8e8e8"),
    yaxis=dict(title="[%]", range=[0, 100], tickvals=list(range(0, 101, 10)),
               showgrid=True, gridcolor="#e8e8e8"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=12)),
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    width=1100, height=500, hovermode="x unified",
)
fig_comp.show()


## 5 · BDP-Potenzial nach Quartal


In [104]:
# Zielwert: 90. Perzentil des bereinigten Datensatzes (Best-Demonstrated-Practice).
target_oee    = df_line["oee_pct"].quantile(0.90)
oee_potential = target_oee - overall_mean

mask_golden = df_line["oee_pct"] >= target_oee
df_standard = df_line[~mask_golden]
df_golden   = df_line[mask_golden]
n_golden    = int(mask_golden.sum())

print(f"Ø OEE (zeitgewichtet):              {overall_mean:.1f} %")
print(f"Ziel-OEE (beste 10 % der Chargen):  {target_oee:.1f} %")
print(f"Ungenutztes Potenzial:              +{oee_potential:.1f} Prozentpunkte")
print(f"Bestleistungs-Chargen: {n_golden} von {len(df_line)} ({n_golden / len(df_line) * 100:.1f} %)")
print()

_df = df_line.copy()
_df["quarter"] = _df["start"].dt.to_period("Q")

bdp_quarterly = (
    _df.groupby("quarter")["oee_pct"]
    .quantile(0.90)
    .reset_index()
    .rename(columns={"oee_pct": "bdp_q90"})
)
bdp_quarterly["q_start"] = bdp_quarterly["quarter"].dt.to_timestamp("D", how="start")
bdp_quarterly["q_end"]   = bdp_quarterly["quarter"].dt.to_timestamp("D", how="end")
bdp_quarterly["label"] = bdp_quarterly.apply(
    lambda r: f"Q{r['quarter'].quarter} {r['quarter'].year}: {r['bdp_q90']:.1f} %", axis=1
)

# Treppenlinie: jeden Wert zweimal eintragen (Quartalsbeginn + Quartalsende).
step_x, step_y = [], []
for _, row in bdp_quarterly.iterrows():
    step_x += [row["q_start"], row["q_end"]]
    step_y += [row["bdp_q90"], row["bdp_q90"]]

print("Rollierendes BDP-Potenzial (90. Perz. je Quartal):\n")
for _, row in bdp_quarterly.iterrows():
    n = int(_df[_df["quarter"] == row["quarter"]]["oee_pct"].count())
    print(f"  {row['label']}  ({n} Chargen)")
print(f"\n  Stat. Gesamt-BDP: {target_oee:.1f} %  |  Ø OEE: {overall_mean:.1f} %")

fig_roll = go.Figure()

fig_roll.add_trace(go.Scatter(
    x=df_standard["start"], y=df_standard["oee_pct"],
    mode="markers", name="Einzelchargen",
    marker=dict(color="#4C72B0", size=4, opacity=0.22),
    hovertemplate="Datum: %{x|%Y-%m-%d}<br>OEE: %{y:.1f} %<extra>Einzelcharge</extra>",
))
fig_roll.add_trace(go.Scatter(
    x=df_golden["start"], y=df_golden["oee_pct"],
    mode="markers", name=f"Golden Batch (≥ stat. BDP {target_oee:.0f} %)",
    marker=dict(color="#2CA02C", size=4, opacity=0.45),
    hovertemplate="<b>Golden Batch</b><br>Datum: %{x|%Y-%m-%d}<br>OEE: %{y:.1f} %<extra></extra>",
))
fig_roll.add_trace(go.Scatter(
    x=monthly["month_dt"], y=monthly["mean"],
    mode="lines+markers", name="Monatl. Ø OEE",
    line=dict(color="#4C72B0", width=2.0),
    marker=dict(size=6, color="white", line=dict(color="#4C72B0", width=1.8)),
    hovertemplate="<b>%{x|%B %Y}</b><br>Ø OEE: %{y:.1f} %<extra></extra>",
))

# Rollierendes BDP als Treppenlinie (grün, durchgezogen).
fig_roll.add_trace(go.Scatter(
    x=step_x, y=step_y,
    mode="lines", name="Rollierendes BDP (90. Perz. je Quartal)",
    line=dict(color="#2CA02C", width=2.8),
    hoverinfo="skip",
))

# Unsichtbare Hover-Punkte je Quartal mit aussagekräftigem Tooltip.
fig_roll.add_trace(go.Scatter(
    x=bdp_quarterly["q_start"], y=bdp_quarterly["bdp_q90"],
    mode="markers", showlegend=False,
    marker=dict(color="#2CA02C", size=10, opacity=0),
    hovertemplate="<b>%{customdata}</b><extra>Rollierendes BDP</extra>",
    customdata=bdp_quarterly["label"],
))

fig_roll.add_hline(
    y=target_oee,
    line=dict(color="#2CA02C", width=1.5, dash="dash"),
    annotation_text=f"Stat. BDP gesamt: {target_oee:.1f} %",
    annotation_position="bottom right",
    annotation_font=dict(color="#2CA02C", size=11),
)
fig_roll.add_hline(
    y=overall_mean,
    line=dict(color="#4C72B0", width=1.5, dash="dash"),
    annotation_text=f"Ø {LINE_LABEL}: {overall_mean:.1f} %",
    annotation_position="bottom left",
    annotation_font=dict(color="#4C72B0", size=11),
)

fig_roll.update_layout(
    title=dict(
        text=(
            f"Rollierendes BDP-Potenzial — {LINE_LABEL}<br>"
            f"<sup>{n_zero} Leerläufe und {n_invalid} Chargen ohne Output entfernt "
            f"| {len(df_line)} Chargen | Ziel-OEE (stat.): {target_oee:.1f} % "
            f"| Potenzial: +{oee_potential:.1f} PP "
            f"| {n_golden} Golden Batches ({n_golden / len(df_line) * 100:.1f} %) "
            f"| Ziel-OEE wird quartalsweise neu berechnet "
            f"| Mittelwert zeitgewichtet nach Chargendauer</sup>"
        ),
        font=dict(size=16),
    ),
    xaxis=dict(title="Monat", tickformat="%b\n%Y", dtick="M1",
               showgrid=True, gridcolor="#e0e0e0"),
    yaxis=dict(title="OEE [%]", range=[0, 100],
               showgrid=True, gridcolor="#e0e0e0"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    width=1100, height=500, hovermode="x unified",
)
fig_roll.show()

Ø OEE (zeitgewichtet):              44.3 %
Ziel-OEE (beste 10 % der Chargen):  66.1 %
Ungenutztes Potenzial:              +21.8 Prozentpunkte
Bestleistungs-Chargen: 51 von 506 (10.1 %)

Rollierendes BDP-Potenzial (90. Perz. je Quartal):

  Q1 2025: 63.7 %  (128 Chargen)
  Q2 2025: 61.8 %  (77 Chargen)
  Q3 2025: 66.1 %  (125 Chargen)
  Q4 2025: 66.8 %  (101 Chargen)
  Q1 2026: 71.4 %  (75 Chargen)

  Stat. Gesamt-BDP: 66.1 %  |  Ø OEE: 44.3 %


## 6 · Materialanalyse — OEE-Schwankungen nach Materialnummer

**Fragestellungen:**
1. Welche Materialien werden am häufigsten produziert? (Volumen)
2. Wie stark schwankt der OEE je Material? (σ)
3. Gibt es Materialien mit systematisch niedrigem OEE?

Darstellung: **Top 10** Materialien nach Gesamtmenge (absteigend).

In [105]:
from plotly.subplots import make_subplots

MATERIAL_COL = "material_number"
TOP_N        = 10

# Stammdaten laden: Materialnummer → Klarname.
master_path = csv_path.parent / "Master Data_2026-03-26-0939.csv"
master_df = pd.read_csv(master_path, encoding="utf-8-sig")
master_df["MATERIAL_NUMBER"] = master_df["MATERIAL_NUMBER"].astype(str).str.strip().str.lstrip("0")
mat_name_map = dict(zip(master_df["MATERIAL_NUMBER"], master_df["MATERIAL_DESCRIPTION"]))

df_mat = df_line.copy()
n_missing = df_mat[MATERIAL_COL].isna().sum()
if n_missing > 0:
    print(f"  {n_missing} Chargen ohne Materialnummer ausgeschlossen.")
df_mat = df_mat.dropna(subset=[MATERIAL_COL])
df_mat[MATERIAL_COL] = df_mat[MATERIAL_COL].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# Zeitgewichtete Aggregation je Materialnummer.
mat_stats = (
    df_mat.groupby(MATERIAL_COL)
    .apply(lambda x: pd.Series({
        "chargen":    len(x),
        "oee_mean":   np.average(x["oee_pct"],          weights=x["duration_min"]),
        "oee_std":    x["oee_pct"].std(),
        "avail_mean": np.average(x["availability_pct"], weights=x["duration_min"]),
        "perf_mean":  np.average(x["performance_pct"],  weights=x["duration_min"]),
        "qual_mean":  np.average(x["quality_pct"],      weights=x["duration_min"]),
        "qty_total":  x["total_quantity"].sum(),
    }))
    .reset_index()
)
mat_stats["oee_std"] = mat_stats["oee_std"].fillna(0)
mat_stats["chargen"] = mat_stats["chargen"].astype(int)
mat_stats = mat_stats.sort_values("qty_total", ascending=False).reset_index(drop=True)

# Klarnamen aus Stammdaten zuordnen.
mat_stats["material_name"] = mat_stats[MATERIAL_COL].map(mat_name_map).fillna(mat_stats[MATERIAL_COL])

mat_top = mat_stats.head(TOP_N).copy()

overall_mat = np.average(df_mat["oee_pct"], weights=df_mat["duration_min"])

print(f"Materialanalyse — {LINE_LABEL}")
print(f"Gesamt-Ø OEE (zeitgewichtet): {overall_mat:.2f} %")
print(f"Eindeutige Materialien: {len(mat_stats)}")
print(f"Darstellung: Top {TOP_N} nach Gesamtmenge (absteigend)\n")

disp = mat_top[["material_name", "chargen", "qty_total",
                "oee_mean", "oee_std",
                "avail_mean", "perf_mean", "qual_mean"]].copy()
disp.columns = ["Material", "Chargen", "Gesamtmenge",
                "Ø OEE", "σ OEE",
                "Ø Verfügb.", "Ø Leistung", "Ø Qualität"]
for col in disp.columns[3:]:
    disp[col] = disp[col].round(1)
print(disp.to_string(index=False))

# Auffälligkeiten im Top-10-Feld markieren.
print("\n── Auffälligkeiten (innerhalb Top 10) ───────────────────────────────")
low_thresh = mat_top["oee_mean"].quantile(0.25)
high_std   = mat_top["oee_std"].quantile(0.75)

low_oee = mat_top[mat_top["oee_mean"] <= low_thresh]
if not low_oee.empty:
    print(f"\n  Niedriger Ø OEE (≤ {low_thresh:.1f} %, unteres Quartil):")
    for _, r in low_oee.iterrows():
        print(f"    {r['material_name']:<40}  Chargen: {int(r['chargen']):>4}  "
              f"Ø OEE: {r['oee_mean']:.1f} %  σ: {r['oee_std']:.1f} %")

vol_high_std = mat_top[mat_top["oee_std"] >= high_std]
if not vol_high_std.empty:
    print(f"\n  Hohe OEE-Schwankung (σ ≥ {high_std:.1f} %, oberes Quartil):")
    for _, r in vol_high_std.iterrows():
        print(f"    {r['material_name']:<40}  Chargen: {int(r['chargen']):>4}  "
              f"Ø OEE: {r['oee_mean']:.1f} %  σ: {r['oee_std']:.1f} %")

Materialanalyse — Abfüllanlage 4-E2G
Gesamt-Ø OEE (zeitgewichtet): 44.25 %
Eindeutige Materialien: 290
Darstellung: Top 10 nach Gesamtmenge (absteigend)

                    Material  Chargen  Gesamtmenge  Ø OEE  σ OEE  Ø Verfügb.  Ø Leistung  Ø Qualität
 SA Coated Beads E2G 14,1 ml       15    1381723.0   53.4    9.4        62.1        88.5        97.0
 SA Coated Beads E2G 12,4 ml       13     749660.0   50.2   13.7        59.3        86.9        97.4
       Leerflasche ePack E2G        3     583366.0   45.2    5.2        48.5        93.9        99.0
              B12 E2G 300 R2        5     546746.0   54.9    9.8        60.3        92.4        99.0
           TSH E2G 300 R2 V2        4     537618.0   52.1    4.9        57.6        91.1        99.2
Vit.D total Gen3 E2G 300 PT2        4     507973.0   58.1    5.0        62.6        94.0        98.8
       HIV Duo M1 E2G 300 R1        5     462297.0   62.6    8.8        68.0        93.1        99.0
 Vit.D total Gen3 E2G 300 R1        6 

### Materialanalyse — Visualisierung

Drei gestapelte Teildiagramme (gemeinsame X-Achse = Materialnummer):
1. **Gesamtmenge** je Material → Produktionsvolumen
2. **Boxplot OEE** je Material → Leistungsniveau & Streuung
3. **Heatmap OEE-Komponenten** → Wo liegt der Verlust je Material?

In [106]:
labels       = mat_top["material_name"].tolist()
qty_vals     = mat_top["qty_total"].tolist()

fig_mat = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        f"1. Produktionsvolumen — Top {TOP_N} Materialien (Gesamtmenge)",
        "2. OEE-Verteilung je Material — Boxplot (Median, Quartile, Ausreißer)",
        "3. OEE-Komponenten je Material (Ø [%])",
    ),
    row_heights=[0.22, 0.42, 0.36],
)

# 1 Volumen-Balken.
fig_mat.add_trace(
    go.Bar(
        x=labels, y=qty_vals,
        name="Gesamtmenge", marker_color="#4C72B0", opacity=0.82,
        text=[f"{v:,.0f}" for v in qty_vals], textposition="outside",
        textfont=dict(size=11, color="#333333"),
        hovertemplate=(
            "<b>Material %{x}</b><br>Gesamtmenge: %{y:,.0f}<br>"
            "Chargen: %{customdata}<extra></extra>"
        ),
        customdata=mat_top["chargen"].tolist(),
    ), row=1, col=1,
)

# 2 Boxplot OEE-Verteilung.
mat_numbers_top = mat_top[MATERIAL_COL].tolist()
df_box = df_mat[df_mat[MATERIAL_COL].isin(mat_numbers_top)].copy()
df_box["material_name"] = df_box[MATERIAL_COL].map(mat_name_map).fillna(df_box[MATERIAL_COL])
df_box["material_name"] = pd.Categorical(df_box["material_name"], categories=labels, ordered=True)
df_box = df_box.sort_values("material_name")

fig_mat.add_trace(
    go.Box(
        x=df_box["material_name"].astype(str), y=df_box["oee_pct"],
        name="OEE-Verteilung",
        marker_color="#4C72B0", line_color="#4C72B0",
        fillcolor="rgba(76, 114, 176, 0.25)",
        boxpoints="outliers", jitter=0,
        marker=dict(size=4, opacity=0.5),
        hovertemplate="<b>Material %{x}</b><br>OEE: %{y:.1f} %<extra></extra>",
    ), row=2, col=1,
)
fig_mat.add_hline(
    y=overall_mat,
    line=dict(color="#C44E52", width=1.8, dash="dash"),
    annotation_text=f"Gesamt-Ø {overall_mat:.1f} %",
    annotation_position="top right",
    annotation_font=dict(color="#C44E52", size=11),
    row=2, col=1,
)

# 3 Heatmap OEE-Komponenten.
comp_matrix = [
    mat_top["avail_mean"].round(1).tolist(),
    mat_top["perf_mean"].round(1).tolist(),
    mat_top["qual_mean"].round(1).tolist(),
]
fig_mat.add_trace(
    go.Heatmap(
        z=comp_matrix, x=labels,
        y=["Verfügbarkeit", "Leistungsgrad", "Qualität"],
        colorscale="RdYlGn", zmin=0, zmax=100,
        text=[[f"{v:.1f}" for v in row] for row in comp_matrix],
        texttemplate="%{text}",
        hovertemplate="<b>Material %{x}</b><br>%{y}: %{z:.1f} %<extra></extra>",
        showscale=True,
        colorbar=dict(title="%", thickness=12, len=0.30, y=0.09, yanchor="bottom"),
    ), row=3, col=1,
)

fig_mat.update_layout(
    title=dict(
        text=(
            f"Materialanalyse — OEE nach Materialnummer | {LINE_LABEL}<br>"
            f"<sup>Top {TOP_N} nach Gesamtmenge (absteigend) · "
            f"{len(mat_stats)} Materialien gesamt · "
            f"Gesamt-Ø OEE: {overall_mat:.1f} % · "
            f"Mittelwert zeitgewichtet nach Chargendauer</sup>"
        ),
        font=dict(size=16),
    ),
    height=980, width=1100,
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    showlegend=False, hovermode="x unified", bargap=0.22,
)

fig_mat.update_yaxes(
    title_text="Gesamtmenge",
    range=[0, max(qty_vals) * 1.18],
    showgrid=True, gridcolor="#e8e8e8", row=1, col=1,
)
fig_mat.update_yaxes(
    title_text="OEE [%]", range=[0, 100],
    showgrid=True, gridcolor="#e8e8e8", row=2, col=1,
)
fig_mat.update_yaxes(title_text="", row=3, col=1)
fig_mat.update_xaxes(
    title_text="Material",
    tickangle=-40, tickfont=dict(size=9), row=3, col=1,
)

fig_mat.show()

## 7 · Schichtanalyse

**Methodik:**
- Jede Charge wird der Schicht zugeordnet, in der sie **gestartet** wurde (`batch_started = 1`).  
  Chargen ohne Startzeile im Datensatz → Fallback auf erste vorhandene Fact-Zeile.
- 3 Schichten sind als GUIDs kodiert — Klarnamen aus Stammdaten zugeordnet.

In [107]:

# ── Dateipfade ────────────────────────────────────────────────────────────────
fact_path = csv_path.parent / "[OEECore.PenzDia.DW].[dbo].[fact].csv"
lj_path   = csv_path.parent / "[OEECore.PenzDia.DW].[dbo].[vw_batch_kpis]_LeftJoin_[vw_batch_status_duration].csv"

fact_raw = pd.read_csv(fact_path, sep=";", decimal=",", encoding="utf-8-sig", low_memory=False)

# Spaltennamen manuell setzen — SQL-Export enthält keinen Header.
LJ_COLS = [
    "batch_number", "order_number", "material_number", "material_name",
    "line_name", "start", "stop",
    "oee2", "availability", "performance", "quality", "capacity_utilization",
    "total_quantity", "good_quantity", "rejects", "target_quantity",
    "runrate_per_minute", "actual_operating_time_min", "actual_runtime_min",
    "theoretical_runtime_min", "uptime_min",
    "planned_downtime_min", "unplanned_downtime_min", "equipment_breakdown_min",
    "changeover_time_min", "cleaning_min", "organizational_downtime_min",
    "non_technical_breakdown_min",
    "batch_oid", "level", "path", "classification_type",
    "classification_type_code", "duration_min", "aggregated_duration_min",
    "aggregated_duration_pct_of_actual_operating",
]
lj_raw = pd.read_csv(
    lj_path, sep=";", decimal=",", encoding="utf-8-sig",
    header=None, names=LJ_COLS, low_memory=False,
)

# ── Schicht-Labels ─────────────────────────────────────────────────────────────
# 3 Schicht-GUIDs — alphabetisch sortiert und Klarnamen zugeordnet.
shift_guids = sorted(fact_raw["key_shift"].dropna().unique())
_shift_names = ["Frühschicht", "Spätschicht", "Nachtschicht"]
SHIFT_LABELS = {guid: _shift_names[i] for i, guid in enumerate(shift_guids)}

# Feste Darstellungsreihenfolge: Früh → Spät → Nacht.
SHIFT_ORDER  = ["Frühschicht", "Spätschicht", "Nachtschicht"]
SHIFT_COLORS = {
    "Frühschicht":  {"line": "#4C72B0", "fill": "rgba(76,114,176,0.15)"},
    "Spätschicht":  {"line": "#DD8452", "fill": "rgba(221,132,82,0.15)"},
    "Nachtschicht": {"line": "#55A868", "fill": "rgba(85,168,104,0.15)"},
}

print("Schicht-Zuordnung (alphabetisch nach GUID):")
for guid, label in SHIFT_LABELS.items():
    n = int((fact_raw["key_shift"] == guid).sum())
    print(f"  {label}: …{guid[-12:]}  ({n} Fact-Zeilen)")

# ── Batch → Primärschicht aus fact.csv ────────────────────────────────────────
# Priorität 1: batch_started=1  → Charge wurde in dieser Schicht gestartet.
# Priorität 2: erste Fact-Zeile je Charge (Fallback für Chargen ohne Startzeile).
batch_fact = fact_raw[fact_raw["key_batch"].notna()].copy()
batch_fact["shift_label"] = batch_fact["key_shift"].map(SHIFT_LABELS)

started  = (batch_fact[batch_fact["batch_started"] == 1]
            .groupby("key_batch")["shift_label"].first()
            .rename("primary_shift"))

fallback = (batch_fact[~batch_fact["key_batch"].isin(started.index)]
            .groupby("key_batch")["shift_label"].first()
            .rename("primary_shift"))

shift_map = pd.concat([started, fallback]).reset_index()
shift_map.columns = ["batch_oid", "primary_shift"]

print(f"\nSchichtzuordnung: {len(shift_map)} Chargen  "
      f"({len(started)} via batch_started=1, {len(fallback)} via Fallback)")
for _s, _n in shift_map["primary_shift"].value_counts().sort_index().items():
    print(f"  {_s}: {_n} Chargen")

# ── OEE-Werte aus LeftJoin — ein Eintrag je Charge ────────────────────────────
# Der Join erzeugt mehrere Zeilen pro Charge (eine je Stillstandsklassifikation).
# Für OEE/Mengen/Dauer genügt die erste Zeile je Charge.
# Betriebsdauer aus actual_operating_time_min (Quellsystem).
for _c in ["oee2", "availability", "performance", "quality", "total_quantity",
           "actual_operating_time_min"]:
    lj_raw[_c] = pd.to_numeric(lj_raw[_c], errors="coerce")

lj_raw["start_dt"] = pd.to_datetime(lj_raw["start"], errors="coerce")

fj_oee = (lj_raw.sort_values("start_dt")
                .groupby("batch_oid")
                .agg(
                    date      = ("start_dt",  "first"),
                    oee_raw   = ("oee2",       "first"),
                    avail_raw = ("availability","first"),
                    perf_raw  = ("performance", "first"),
                    qual_raw  = ("quality",     "first"),
                    qty_total = ("total_quantity", "first"),
                    dur_min   = ("actual_operating_time_min", "first"),
                )
                .reset_index())

# ── Join: OEE + Schicht ────────────────────────────────────────────────────────
# Inner Join: nur Chargen, die sowohl im LeftJoin-Datensatz als auch in fact.csv vorkommen.
# Chargen ohne Schicht-Eintrag in fact.csv (z. B. zu alte Daten) fallen hier heraus.
n_lj_batches = len(fj_oee)
df_shift = fj_oee.merge(shift_map, on="batch_oid", how="inner")
n_drop_merge = n_lj_batches - len(df_shift)

# Bereinigung analog zu df_line — mit Zählung der entfernten Chargen je Grund.
n_before = len(df_shift)
df_shift = df_shift.dropna(subset=["oee_raw", "date"]).copy()
n_drop_nan = n_before - len(df_shift)

n_before = len(df_shift)
df_shift = df_shift[df_shift["qty_total"] > 0].copy()
n_drop_qty = n_before - len(df_shift)

n_before = len(df_shift)
df_shift = df_shift[(df_shift["oee_raw"] > 0) & (df_shift["oee_raw"] <= 1)].copy()
n_drop_oee = n_before - len(df_shift)

n_before = len(df_shift)
df_shift = df_shift[df_shift["dur_min"].notna() & (df_shift["dur_min"] > 0)].copy()
n_drop_dur = n_before - len(df_shift)

# Chargen ohne gültige Schichtzuordnung entfernen.
n_before = len(df_shift)
df_shift = df_shift.dropna(subset=["primary_shift"]).copy()
n_drop_shift = n_before - len(df_shift)

df_shift["oee_pct"]   = (df_shift["oee_raw"]   * 100).clip(upper=100)
df_shift["avail_pct"] = (df_shift["avail_raw"]  * 100).clip(upper=100)
df_shift["perf_pct"]  = (df_shift["perf_raw"]   * 100).clip(upper=100)
df_shift["qual_pct"]  = (df_shift["qual_raw"]   * 100).clip(upper=100)
df_shift["month"]     = df_shift["date"].dt.to_period("M")

if df_shift["dur_min"].sum() > 0:
    overall_shift_mean = np.average(df_shift["oee_pct"], weights=df_shift["dur_min"])
else:
    overall_shift_mean = df_shift["oee_pct"].mean()

print(f"\nSchichtanalyse: {n_lj_batches} Chargen im LeftJoin-Datensatz → {len(df_shift)} bereinigte Chargen")
if n_drop_merge > 0:
    print(f"  − {n_drop_merge} entfernt: kein Schicht-Eintrag in fact.csv")
if n_drop_nan > 0:
    print(f"  − {n_drop_nan} entfernt: fehlender OEE oder Startdatum (NaN)")
if n_drop_qty > 0:
    print(f"  − {n_drop_qty} entfernt: keine Produktionsmenge (Leerläufe, qty_total = 0)")
if n_drop_oee > 0:
    print(f"  − {n_drop_oee} entfernt: ungültiger OEE-Wert (≤ 0 oder > 1)")
if n_drop_dur > 0:
    print(f"  − {n_drop_dur} entfernt: keine gültige Betriebsdauer (actual_operating_time_min ≤ 0 oder NaN)")
if n_drop_shift > 0:
    print(f"  − {n_drop_shift} entfernt: keine gültige Schichtzuordnung")
print()
for _s in SHIFT_ORDER:
    _sub   = df_shift[df_shift["primary_shift"] == _s]
    if _sub.empty:
        continue
    if _sub["dur_min"].sum() > 0:
        _wmean = np.average(_sub["oee_pct"], weights=_sub["dur_min"])
    else:
        _wmean = _sub["oee_pct"].mean()
    print(f"  {_s}: {len(_sub):>4} Chargen | Ø OEE (zeitgew.): {_wmean:.1f} %")

print(f"\n  Gesamt-Ø OEE (alle Schichten): {overall_shift_mean:.1f} %")

Schicht-Zuordnung (alphabetisch nach GUID):
  Frühschicht: …32AADD1FA3DA  (1413 Fact-Zeilen)
  Spätschicht: …73377968E9E7  (1420 Fact-Zeilen)
  Nachtschicht: …3885FCA3A42D  (1398 Fact-Zeilen)

Schichtzuordnung: 578 Chargen  (578 via batch_started=1, 0 via Fallback)
  Frühschicht: 184 Chargen
  Nachtschicht: 193 Chargen
  Spätschicht: 201 Chargen

Schichtanalyse: 548 Chargen im LeftJoin-Datensatz → 533 bereinigte Chargen
  − 15 entfernt: kein Schicht-Eintrag in fact.csv

  Frühschicht:  184 Chargen | Ø OEE (zeitgew.): 50.3 %
  Spätschicht:  181 Chargen | Ø OEE (zeitgew.): 38.5 %
  Nachtschicht:  168 Chargen | Ø OEE (zeitgew.): 43.3 %

  Gesamt-Ø OEE (alle Schichten): 44.0 %


In [108]:

# Zeitgewichteter monatlicher Mittelwert je Schicht.
monthly_shift = {}
for _s, _grp in df_shift.groupby("primary_shift"):
    _agg = (
        _grp.groupby("month")
        .apply(lambda x: pd.Series({
            "mean":  np.average(x["oee_pct"], weights=x["dur_min"]) if x["dur_min"].sum() > 0 else x["oee_pct"].mean(),
            "count": len(x),
        }))
        .reset_index()
    )
    _agg["month_dt"] = _agg["month"].dt.to_timestamp()
    monthly_shift[_s] = _agg

fig_shift_trend = go.Figure()

# Einzelchargen als sehr kleine graue Punkte im Hintergrund.
fig_shift_trend.add_trace(go.Scatter(
    x=df_shift["date"], y=df_shift["oee_pct"],
    mode="markers", name="Einzelchargen",
    marker=dict(color="#bbbbbb", size=3, opacity=0.18),
    hoverinfo="skip",
))

for _s in SHIFT_ORDER:
    if _s not in monthly_shift:
        continue
    _cfg  = SHIFT_COLORS[_s]
    _agg  = monthly_shift[_s]
    _sub  = df_shift[df_shift["primary_shift"] == _s]
    _wmean = np.average(_sub["oee_pct"], weights=_sub["dur_min"]) if _sub["dur_min"].sum() > 0 else _sub["oee_pct"].mean()
    _n     = len(_sub)
    fig_shift_trend.add_trace(go.Scatter(
        x=_agg["month_dt"], y=_agg["mean"],
        mode="lines+markers",
        name=f"{_s}  (Ø {_wmean:.1f} %,  n={_n})",
        line=dict(color=_cfg["line"], width=2.2),
        marker=dict(size=7, color="white", line=dict(color=_cfg["line"], width=2)),
        hovertemplate=(
            f"<b>%{{x|%B %Y}}</b><br>{_s}<br>"
            "Ø OEE: %{y:.1f} %<br>Chargen: %{customdata}<extra></extra>"
        ),
        customdata=_agg["count"],
    ))

fig_shift_trend.add_hline(
    y=overall_shift_mean,
    line=dict(color="#888888", width=1.5, dash="dash"),
    annotation_text=f"Gesamt-Ø: {overall_shift_mean:.1f} %",
    annotation_position="top right",
    annotation_font=dict(color="#555555", size=11),
)

fig_shift_trend.update_layout(
    title=dict(
        text=(
            f"Monatlicher OEE-Verlauf nach Schicht — {LINE_LABEL}<br>"
            f"<sup>{len(df_shift)} bereinigte Chargen | Mittelwert zeitgewichtet nach Chargendauer</sup>"
        ),
        font=dict(size=16),
    ),
    xaxis=dict(title="Monat", tickformat="%b\n%Y", dtick="M1",
               showgrid=True, gridcolor="#e0e0e0"),
    yaxis=dict(title="OEE [%]", range=[0, 100],
               showgrid=True, gridcolor="#e0e0e0"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    width=1100, height=480, hovermode="x unified",
)

fig_shift_trend.show()

In [109]:

# Stillstandszeiten aus fact.csv (Zeilen ohne Charge = reine Schicht-Ereignisse).
fact_raw["shift_label"] = fact_raw["key_shift"].map(SHIFT_LABELS)
dt_raw = fact_raw[fact_raw["key_batch"].isna()].copy()
for _c in ["planned_downtime_min", "unplanned_downtime_min"]:
    dt_raw[_c] = pd.to_numeric(dt_raw[_c], errors="coerce").fillna(0)

dt_per_shift = (dt_raw.groupby("shift_label")
                .agg(planned   = ("planned_downtime_min",   "sum"),
                     unplanned = ("unplanned_downtime_min", "sum"))
                .reset_index())
dt_per_shift["shift_label"] = pd.Categorical(dt_per_shift["shift_label"], categories=SHIFT_ORDER, ordered=True)
dt_per_shift = dt_per_shift.sort_values("shift_label")

fig_shift_box = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.14,
    row_heights=[0.55, 0.45],
)

# ① Boxplot OEE je Schicht.
for _s in SHIFT_ORDER:
    _sub = df_shift[df_shift["primary_shift"] == _s]
    if _sub.empty:
        continue
    _cfg = SHIFT_COLORS[_s]
    fig_shift_box.add_trace(
        go.Box(
            y=_sub["oee_pct"], name=_s,
            marker_color=_cfg["line"], line_color=_cfg["line"],
            fillcolor=_cfg["fill"],
            boxpoints="outliers", jitter=0,
            marker=dict(size=4, opacity=0.5),
            hovertemplate=f"<b>{_s}</b><br>OEE: %{{y:.1f}} %<extra></extra>",
        ), row=1, col=1,
    )

fig_shift_box.add_hline(
    y=overall_shift_mean,
    line=dict(color="#888888", width=1.5, dash="dash"),
    annotation_text=f"Gesamt-Ø {overall_shift_mean:.1f} %",
    annotation_position="top right",
    annotation_font=dict(color="#555555", size=11),
    row=1, col=1,
)

# ② Stillstand gestapelt (geplant + ungeplant).
fig_shift_box.add_trace(
    go.Bar(
        x=dt_per_shift["shift_label"], y=dt_per_shift["planned"],
        name="Geplanter Stillstand", marker_color="#4C72B0", opacity=0.82,
        text=dt_per_shift["planned"].round(0).astype(int),
        textposition="inside",
        hovertemplate="<b>%{x}</b><br>Geplant: %{y:.0f} min<extra></extra>",
    ), row=2, col=1,
)
fig_shift_box.add_trace(
    go.Bar(
        x=dt_per_shift["shift_label"], y=dt_per_shift["unplanned"],
        name="Ungeplanter Stillstand", marker_color="#C44E52", opacity=0.82,
        text=dt_per_shift["unplanned"].round(0).astype(int),
        textposition="inside",
        hovertemplate="<b>%{x}</b><br>Ungeplant: %{y:.0f} min<extra></extra>",
    ), row=2, col=1,
)

fig_shift_box.update_layout(
    title=dict(
        text=f"Schichtvergleich — OEE-Verteilung & Stillstandszeiten | {LINE_LABEL}",
        font=dict(size=16),
    ),
    barmode="stack",
    height=780, width=1000,
    plot_bgcolor="white", paper_bgcolor="#f8f8f8",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="closest",
)
fig_shift_box.update_yaxes(title_text="OEE [%]", range=[0, 100],
                            showgrid=True, gridcolor="#e8e8e8", row=1, col=1)
fig_shift_box.update_yaxes(title_text="Minuten [min]",
                            showgrid=True, gridcolor="#e8e8e8", row=2, col=1)
fig_shift_box.show()